# 23 FTP and Copay: Phase 1 core features (C1 - C6, gate G1)

One combined notebook (D6) over the shared intermediate `ftp_copay_fill_events`. Builds the 27-column core block for every universe HCP x promotional cohort (c1 - c28) from the UC claims tables, by fill date, one count per claim, point-in-time (DEC-02), then runs the G1 checks. Replaces the 3.0 `19_ftp_features` / `22_copay_features` logic (written date, record grain, running totals); those notebooks are kept for history only. New column names (`FTP_*` / `CPY_*`), never `TOTAL_RX_FTP_IN_LAST_*` (D6).

In [0]:
%run ./00_ftp_copay_config

In [0]:
import pandas as pd
# O1: fail loudly when the source is stale
for p, t in PROGRAM_TABLES.items():
    s = spark.sql(f"SELECT DATEDIFF(current_date(), MAX(REFRESH_TS)) r, DATEDIFF(current_date(), MAX(INSERT_TS)) i FROM {t}").first()
    assert s.r <= REFRESH_MAX_AGE_DAYS, f"{p}: REFRESH_TS is {s.r} days old (> {REFRESH_MAX_AGE_DAYS})"
    assert s.i <= SOURCE_STALE_DAYS, f"{p}: source INSERT_TS is {s.i} days old (> {SOURCE_STALE_DAYS})"

events = build_fill_events()
events.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FILL_EVENTS_PATH)
events = spark.read.format("delta").load(FILL_EVENTS_PATH)
display(events.groupBy("PROGRAM").agg(F.count("*").alias("rows"), F.countDistinct("ORGN_CLM_AUTH_NUM").alias("claims"),
                                      F.countDistinct("PROVIDER_ID").alias("hcps"), F.min("RX_FILL_DATE"), F.max("RX_FILL_DATE")))

In [0]:
cohorts = load_cohorts()
features = assemble_full(build_core_features(events, cohorts), cohorts)
features.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(FEATURES_PATH)
features = spark.read.format("delta").load(FEATURES_PATH)
print(f"{FEATURES_PATH}: {features.count():,} rows x {len(features.columns)} columns")
display(features.filter("FTP_HAS_ANY_12M = 1 AND CPY_HAS_ANY_12M = 1 AND COHORT = 'c28'").limit(5))

## Gate G1 checks
Fixed schema from config; zero nulls and duplicate keys; full universe x cohort; per-cohort coverage vs the assessment (S4 on the claims window is reconciled in 61 V2; here the promotional c28 values are compared with a 2% tolerance); invariants: windows non-decreasing, new-Rx <= fills, has-any = (12M fills > 0), sentinel used exactly when no fill was ever known, caps and bands consistent, C6 = both flags, C5 zero when no copay fill; and a second build gives identical rows.

In [0]:
G1 = []
def chk(name, ok, ev):
    G1.append({"id": "G1", "check": name, "status": "PASS" if ok else "FAIL", "evidence": str(ev)}); print(f"[{'PASS' if ok else 'FAIL'}] {name}: {ev}")

fc = feature_columns()
chk("Schema = config", features.columns == KEYS + fc + ["PULL_DT"], f"{len(fc)} features")
n_u = spark.read.parquet(UNIVERSE_PATH).select("PROVIDER_ID").distinct().count(); n_c = cohorts.count()
n = features.count(); n_keys = features.select("HCP_COHORT_ID").distinct().count()
chk("Full universe x cohort, no duplicate keys", n == n_u * n_c == n_keys, f"{n:,} rows = {n_u:,} x {n_c}; distinct keys {n_keys:,}")
nulls = features.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in KEYS + fc]).first().asDict()
chk("Zero nulls", sum(nulls.values()) == 0, {k: v for k, v in nulls.items() if v})

inv = []
for p in ("FTP", "CPY"):
    inv += [(f"{p} windows non-decreasing", f"NOT ({p}_FILLS_1M <= {p}_FILLS_3M AND {p}_FILLS_3M <= {p}_FILLS_6M AND {p}_FILLS_6M <= {p}_FILLS_12M)"),
            (f"{p} new-Rx windows non-decreasing", f"NOT ({p}_NEWRX_FILLS_1M <= {p}_NEWRX_FILLS_3M AND {p}_NEWRX_FILLS_3M <= {p}_NEWRX_FILLS_6M AND {p}_NEWRX_FILLS_6M <= {p}_NEWRX_FILLS_12M)"),
            (f"{p} new-Rx <= fills", " OR ".join(f"{p}_NEWRX_FILLS_{w}M > {p}_FILLS_{w}M" for w in WINDOWS_MONTHS)),
            (f"{p} has-any = fills_12M > 0", f"{p}_HAS_ANY_12M <> CAST({p}_FILLS_12M > 0 AS INT)"),
            (f"{p} cap respected", f"{p}_FILLS_12M_CAP <> LEAST({p}_FILLS_12M, {CAPS_12M[p]})"),
            (f"{p} band 0 iff no 12M fill", f"({p}_FILLS_12M_BAND = 0) <> ({p}_FILLS_12M = 0)"),
            (f"{p} recency sentinel or within history", f"NOT ({p}_DAYS_SINCE_LAST_FILL = {RECENCY_SENTINEL_DAYS} OR {p}_DAYS_SINCE_LAST_FILL BETWEEN 0 AND 3000)"),
            (f"{p} recency <= window when filled in 12M", f"{p}_HAS_ANY_12M = 1 AND {p}_DAYS_SINCE_LAST_FILL > 366"),
            (f"{p} recency > 12M when no 12M fill", f"{p}_HAS_ANY_12M = 0 AND {p}_DAYS_SINCE_LAST_FILL <= 300")]
inv += [("C6 = both flags", "FTP_CPY_BOTH_12M <> FTP_HAS_ANY_12M * CPY_HAS_ANY_12M"),
        ("C5 zero without copay fill", "CPY_HAS_ANY_12M = 0 AND (CPY_AVG_COPAY_BEFORE_CARD_12M <> 0 OR CPY_COPAY_GT300_SHARE_12M <> 0)"),
        ("C5 share in [0,1], avg in valid range", f"CPY_COPAY_GT300_SHARE_12M NOT BETWEEN 0 AND 1 OR CPY_AVG_COPAY_BEFORE_CARD_12M NOT BETWEEN {COPAY_VALID_RANGE[0]} AND {COPAY_VALID_RANGE[1]}")]
viol = features.select([F.sum(F.expr(f"CASE WHEN {e} THEN 1 ELSE 0 END")).alias(f"v{i}") for i, (_, e) in enumerate(inv)]).first()
for i, (name, _) in enumerate(inv):
    chk(f"Invariant: {name}", viol[f"v{i}"] == 0, f"{viol[f'v{i}']} violations")

# Coverage vs the assessment (promotional c28 vs assessment claims-window c28; tolerance 2%)
c28 = features.filter("COHORT = 'c28'").agg(F.sum("FTP_HAS_ANY_12M").alias("ftp_h"), F.sum("FTP_FILLS_12M").alias("ftp_f"),
                                            F.sum("CPY_HAS_ANY_12M").alias("cpy_h"), F.sum("CPY_FILLS_12M").alias("cpy_f"),
                                            F.sum("FTP_CPY_BOTH_12M").alias("both")).first().asDict()
exp = {"ftp_h": 3951, "ftp_f": 9279, "cpy_h": 4992, "cpy_f": 32057, "both": 1054}
dev = {k: round(100 * (c28[k] - exp[k]) / exp[k], 2) for k in exp}
chk("Coverage c28 vs assessment (within 2%)", all(abs(v) <= 2 for v in dev.values()), f"c28 {c28}; % deviation {dev}")

# Reproducibility: second build identical (S5)
rebuild = assemble_full(build_core_features(events, cohorts), cohorts).drop("PULL_DT")
diff = rebuild.exceptAll(features.drop("PULL_DT")).count() + features.drop("PULL_DT").exceptAll(rebuild).count()
chk("Second build identical", diff == 0 and rebuild.columns == features.drop("PULL_DT").columns, f"{diff} differing rows")

g1 = all(r["status"] == "PASS" for r in G1)
print(f"G1: {sum(r['status'] == 'PASS' for r in G1)} of {len(G1)} checks PASS -> G1 = {'PASS' if g1 else 'FAIL'}")
save_qc("phase1_g1_checks", G1 + [{"id": "G1", "check": "Gate G1", "status": "PASS" if g1 else "FAIL", "evidence": f"{len(G1)} checks"}])

In [0]:
cov = (features.groupBy("COHORT", "DATASET").agg(
          F.round(100 * F.avg(F.greatest("FTP_HAS_ANY_12M", "CPY_HAS_ANY_12M")), 2).alias("PCT_ANY"),
          F.sum("FTP_HAS_ANY_12M").alias("FTP_HCPS_12M"), F.sum("FTP_FILLS_12M").alias("FTP_FILLS_12M"),
          F.sum("CPY_HAS_ANY_12M").alias("CPY_HCPS_12M"), F.sum("CPY_FILLS_12M").alias("CPY_FILLS_12M"),
          F.sum("FTP_CPY_BOTH_12M").alias("BOTH_HCPS_12M"))
       .withColumn("N", F.expr("cast(substring(COHORT, 2) as int)")).orderBy("N").drop("N"))
display(cov)
fr = features.select([F.round(100 * F.avg((F.col(c) != (RECENCY_SENTINEL_DAYS if "DAYS_SINCE" in c else 0)).cast("int")), 3).alias(c) for c in fc]).toPandas().T
fr.columns = ["PCT_NON_DEFAULT_ALL_COHORTS"]
fr["PCT_NULL"] = 0.0
display(fr.reset_index().rename(columns={"index": "FEATURE"}))

## Feature dictionary (27 columns)

All windows end at E = `SVC_DT` - 14 days and cover (E - N months, E]; a claim counts only if it was loaded (`INSERT_TS`) by `SVC_DT`. One count per original claim (`ORGN_CLM_AUTH_NUM`). `p` = `FTP` (7 WF groups, `kerendia_ftp_claims`) or `CPY` (3 DF groups, `kerendia_copay_claims`).

| Family | Column | Definition | No activity |
| --- | --- | --- | --- |
| C1 | `p_HAS_ANY_12M` | 1 if at least one fill in 12M | 0 |
| C2 | `p_FILLS_{1,3,6,12}M` | claims filled in the window | 0 |
| C2 | `p_FILLS_12M_CAP` | `p_FILLS_12M` capped at the training p99 (FTP 21, copay 53) | 0 |
| C2 | `p_FILLS_12M_BAND` | 0 = none, 1 = 1, 2 = 2-3, 3 = 4-9, 4 = 10-24, 5 = 25+ | 0 |
| C3 | `p_NEWRX_FILLS_{1,3,6,12}M` | fills with `IS_NEW_RX` (new prescription, `NEW_REFILL_CD = '00'`) | 0 |
| C4 | `p_DAYS_SINCE_LAST_FILL` | days from E back to the last known fill (any time since 2021) | 9999 |
| C5 | `CPY_AVG_COPAY_BEFORE_CARD_12M` | mean `NET_PTNT_COPAY` (USD, the copay the patient faced before the card) over 12M copay fills with a value in [0, 2000] | 0 |
| C5 | `CPY_COPAY_GT300_SHARE_12M` | share of those fills with copay > USD 300 | 0 |
| C6 | `FTP_CPY_BOTH_12M` | 1 if the HCP has both an FTP and a copay fill in 12M | 0 |

**Business rules**: UC tables already apply USE_SLS = 'Y', reversal removal, duplicate resolution and clawback netting (not re-derived). Claims without `PRSC_NPI` are excluded. Link universe `PROVIDER_ID` -> NPI -> `PRSC_NPI`. BRIDGE is excluded. Clawback / zero-benefit flags, `MDM_SPECIALTY` and the written date are never used. Money caveat (D9): `NET_*` is netted with clawbacks known at the weekly rebuild, so C5 values for past cohorts can change slightly between rebuilds; counts do not.

**Dropped from 3.0**: `TOTAL_RX_FTP_IN_LAST_*`, `RUNNING_TOTAL_RX_FTP`, copay `PCT_*` ratios and the record-grain / written-date counting.